# 🧠 Umjetna inteligencija na mikroupravljaču
## Radionica: prepoznavanje znamenki na M5StickV

U ovoj bilježnici ćete:

1. **pogledati podatke** – 70 000 rukom pisanih znamenki (skup MNIST),
2. **istrenirati neuronsku mrežu** koja prepoznaje znamenke 0–9,
3. **pretvoriti model** u format koji razumije čip K210 u StickV-u (`.kmodel`),
4. **preuzeti model** i prebaciti ga na uređaj.

👉 Ćelije pokrećete redom: kliknite u ćeliju i pritisnite **Shift + Enter**.

👉 Ćelije označene s ✏️ smijete mijenjati.

## 1. korak – Podaci
Učitavamo skup MNIST: 60 000 slika za učenje i 10 000 za provjeru. Svaka slika ima 28 × 28 piksela.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Pikseli su 0–255, mreža voli brojeve 0–1
x_train = (x_train / 255.0).astype("float32")[..., None]
x_test  = (x_test  / 255.0).astype("float32")[..., None]

print("Slika za učenje:", len(x_train), " | slika za provjeru:", len(x_test))
print("TensorFlow verzija:", tf.__version__)

plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i, :, :, 0], cmap="gray")
    plt.title(str(y_train[i]))
    plt.axis("off")
plt.show()

💡 **Primijetite:** znamenke su **bijele na crnoj podlozi** i nalaze se u sredini slike.
Kamera na StickV-u vidi tamni flomaster na bijelom papiru, zato će uređaj sliku prvo obrnuti i centrirati.

## 2. korak – Neuronska mreža ✏️
Gradimo **konvolucijsku neuronsku mrežu** (CNN). Konvolucijski slojevi traže uzorke (rubove, krivulje, petlje),
zatim se za svaku od 64 značajke izračuna prosjek, a zadnji sloj odlučuje koja je to znamenka.

Možete mijenjati broj filtera. Više filtera = pametnija, ali veća i sporija mreža.

⚠️ Ne dodajte sloj `Flatten`: alat za K210 koji koristimo ga ne podržava. Zato koristimo `GlobalAveragePooling2D`.

In [ ]:
# ✏️ Ovo smijete mijenjati
FILTERI_1 = 16   # probajte 8, 16 ili 32
FILTERI_2 = 32   # probajte 16, 32 ili 64
FILTERI_3 = 64   # probajte 32 ili 64

def napravi_model():
    ulaz = tf.keras.Input(shape=(28, 28, 1), batch_size=1, name="slika")
    x = tf.keras.layers.Conv2D(FILTERI_1, 3, padding="same", activation="relu")(ulaz)
    x = tf.keras.layers.MaxPooling2D(2)(x)                        # 28x28 -> 14x14
    x = tf.keras.layers.Conv2D(FILTERI_2, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)                        # 14x14 -> 7x7
    x = tf.keras.layers.Conv2D(FILTERI_2, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.Conv2D(FILTERI_3, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)               # prosjek svake značajke
    izlaz = tf.keras.layers.Dense(10, activation="softmax")(x)    # 10 znamenki
    return tf.keras.Model(ulaz, izlaz)

model = napravi_model()
model.summary()

## 3. korak – Učenje ✏️
Mrežu učimo na **malo izmijenjenim** slikama: zarotiranima, pomaknutima i uvećanima.
Tako se lakše snalazi sa znamenkama koje vi nacrtate (nikad nisu savršeno u sredini).

Jedna **epoha** znači da je mreža jednom vidjela svih 60 000 slika.

In [ ]:
# ✏️ Ovo smijete mijenjati
EPOHE = 5        # 1 epoha traje oko 15 s; 5 je dobro, 8 još bolje

izmjene = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomTranslation(0.12, 0.12),
    tf.keras.layers.RandomZoom(0.15),
])
model_za_ucenje = tf.keras.Sequential([tf.keras.Input((28, 28, 1)), izmjene, model])
model_za_ucenje.compile(optimizer="adam",
                        loss="sparse_categorical_crossentropy",
                        metrics=["accuracy"])

povijest = model_za_ucenje.fit(x_train, y_train, epochs=EPOHE, batch_size=128,
                               validation_data=(x_test, y_test))

## 4. korak – Koliko je mreža dobra?

In [ ]:
predvidanja = model_za_ucenje.predict(x_test, batch_size=1000, verbose=0)
pogodak = predvidanja.argmax(axis=1) == y_test
print("Točnost na 10 000 novih slika: %.2f %%" % (100 * pogodak.mean()))

# Pogledajmo gdje je mreža pogriješila
krive = np.where(~pogodak)[0][:10]
plt.figure(figsize=(10, 2.4))
for n, i in enumerate(krive):
    plt.subplot(1, 10, n + 1)
    plt.imshow(x_test[i, :, :, 0], cmap="gray")
    plt.title("%d→%d" % (y_test[i], predvidanja[i].argmax()), color="red")
    plt.axis("off")
plt.suptitle("Pogreške (prava → mreža misli)")
plt.show()

💬 **Za razmisliti:** biste li vi točno pročitali te znamenke?

## 5. korak – Pretvorba za mikroupravljač
Čip **Kendryte K210** ima posebnu jezgru za neuronske mreže (KPU). Ona ne računa s decimalnim brojevima,
nego s cijelim brojevima 0–255 (**kvantizacija**). Zato model pretvaramo u dva koraka:

`Keras model` → `.tflite` → `.kmodel` (alat **nncase**)

In [ ]:
import os, subprocess
from PIL import Image

# a) Keras -> TensorFlow Lite
tflite_model = tf.lite.TFLiteConverter.from_keras_model(model).convert()
open("mnist.tflite", "wb").write(tflite_model)
print("mnist.tflite: %.1f KB" % (len(tflite_model) / 1024))

# b) Alat nncase v0.1 (prevodi za K210; radi s firmwareom StickV-a 0.4.0)
if not os.path.exists("nncase/ncc"):
    subprocess.run("wget -q https://github.com/kendryte/nncase/releases/download/v0.1.0-rc5/ncc-linux-x86_64.tar.xz", shell=True)
    os.makedirs("nncase", exist_ok=True)
    subprocess.run("tar -xJf ncc-linux-x86_64.tar.xz -C nncase && chmod +x nncase/ncc", shell=True)

# c) Slike za kalibraciju (nncase iz njih određuje raspon brojeva)
os.makedirs("kalibracija", exist_ok=True)
for i in range(100):
    Image.fromarray((x_train[i, :, :, 0] * 255).astype("uint8")).save("kalibracija/%d.png" % i)

# d) Prevođenje
if os.path.exists("mnist.kmodel"):
    os.remove("mnist.kmodel")
subprocess.run("DOTNET_SYSTEM_GLOBALIZATION_INVARIANT=1 ./nncase/ncc -i tflite -o k210model "
               "--dataset kalibracija mnist.tflite mnist.kmodel > ncc_log.txt 2>&1", shell=True)
if os.path.exists("mnist.kmodel"):
    print("✅ mnist.kmodel: %.1f KB" % (os.path.getsize("mnist.kmodel") / 1024))
else:
    print("❌ Pretvorba nije uspjela. Zovite voditelja. Zadnje linije zapisa:")
    print(open("ncc_log.txt").read()[-1500:])

## 6. korak – Što će raditi KPU, a što procesor?
nncase je zapisao kako je rasporedio slojeve. Slojevi **K210Conv2d** izvode se na KPU-u, posebnoj jezgri za neuronske mreže.
Ostali (npr. *GlobalAveragePool*, *Softmax*) izvode se na običnom procesoru.

In [ ]:
import re
for linija in open("ncc_log.txt").read().splitlines():
    if re.match(r"^\d+: ", linija) or "memory" in linija:
        print(linija)

## 7. korak – Preuzmite model 📥
Pokrenite ćeliju i spremite datoteku **mnist.kmodel** na računalo.
Zatim slijedite upute na papiru: MaixPy IDE → *Send file to board*.

In [ ]:
from google.colab import files
files.download("mnist.kmodel")

## 8. korak – Usporedba: linearni model 📏
Je li za prepoznavanje znamenki uopće potrebna neuronska mreža? Probajmo najjednostavniji mogući model: **linearnu regresiju**.

* Slika 28 × 28 se „razvuče” u **784 broja** (piksela).
* Za svaku znamenku model ima 784 **težine** i jedan pomak. Zbroji *piksel × težina* i dobije broj.
* Pobjeđuje znamenka s najvećim zbrojem.

Ovdje nema epoha: težine se izračunaju **odjednom**, rješavanjem sustava jednadžbi (metoda najmanjih kvadrata).

In [ ]:
X  = x_train.reshape(-1, 784)                 # svaka slika = redak od 784 broja
Xt = x_test.reshape(-1, 784)
A  = np.hstack([X, np.ones((len(X), 1))])     # dodatni stupac jedinica = pomak (bias)
Y  = np.eye(10)[y_train]                      # cilj: npr. 3 -> [0,0,0,1,0,0,0,0,0,0]

# Najmanji kvadrati s regularizacijom: (AᵀA + λI) W = AᵀY
# λ "kažnjava" prevelike težine, pa su težine glađe i lakše ih je razumjeti
LAMBDA = 5000    # ✏️ probajte 10 ili 50000 i pogledajte slike težina
W = np.linalg.solve(A.T @ A + LAMBDA * np.eye(785), A.T @ Y)
print("Broj težina:", W.size)

rez = np.hstack([Xt, np.ones((len(Xt), 1))]) @ W
print("Točnost linearnog modela: %.2f %%" % (100 * (rez.argmax(axis=1) == y_test).mean()))
print("Točnost vaše mreže:       %.2f %%" % (100 * pogodak.mean()))

### Što je linearni model naučio?
Težine svake znamenke možemo nacrtati kao sliku 28 × 28. **Crveno** = piksel tu „glasa za” znamenku, **plavo** = „glasa protiv”.

In [ ]:
plt.figure(figsize=(12, 2.6))
g = np.abs(W[:784]).max()
for z in range(10):
    plt.subplot(1, 10, z + 1)
    plt.imshow(W[:784, z].reshape(28, 28), cmap="bwr", vmin=-g, vmax=g)
    plt.title(str(z)); plt.axis("off")
plt.suptitle("Težine linearnog modela")
plt.show()

💬 **Za razmisliti:** vidite li obrise znamenki? Što će se dogoditi ako znamenku nacrtate u kutu umjesto u sredini?

### Izvoz za StickV
Težine pretvaramo u cijele brojeve od −127 do 127 (1 bajt po težini) i spremamo u **linearni.bin** (oko 8 KB).
Na StickV-u ovaj model ne računa KPU, nego obični procesor.

In [ ]:
mjerilo = 127 / np.abs(W).max()
Wq = np.round(W * mjerilo).astype(np.int8)    # 785 x 10: retci = pikseli (+ pomak), stupci = znamenke
tocno_q = ((np.hstack([Xt, np.ones((len(Xt), 1))]) @ (Wq / mjerilo)).argmax(axis=1) == y_test).mean()
print("Točnost nakon pretvorbe u cijele brojeve: %.2f %%" % (100 * tocno_q))

with open("linearni.bin", "wb") as f:
    f.write(b"LIN1")
    f.write(int(round(mjerilo)).to_bytes(2, "little"))
    f.write(Wq.tobytes())
print("linearni.bin: %d bajtova" % os.path.getsize("linearni.bin"))

from google.colab import files
files.download("linearni.bin")

Pošaljite **linearni.bin** na StickV isto kao i mnist.kmodel (*Tools → Send file to board*) i ponovno ga pokrenite.
Tipkom **B** mijenjate model: **VAS** (vaša mreža) → **REZ** (rezervna mreža) → **LIN** (linearni model).

## 🎯 Izazovi
* Nacrtajte znamenku koju StickV prepozna sa **sigurnošću preko 95 %**.
* Nacrtajte nešto **što nije znamenka** (srce, slovo A, mačku). Što kaže mreža? Zašto?
* Okrenite papir naopako. Prepoznaje li 6 kao 9?
* Usporedite svoj model s rezervnim (tipka **B** na StickV-u). Koji je bolji?
* Nađite znamenku koju mreža prepozna, a linearni model ne. Gdje ste je nacrtali? Kako je nagnuta?
* Vratite se na 2. korak, smanjite filtere na 4, 8 i 16 i istrenirajte opet. Što se dogodilo s točnošću i veličinom modela?